In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser

# 🔐 Load API keys
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path=".env")
google_api_key = os.getenv("GOOGLE_API_KEY")

prompt = PromptTemplate.from_template("Translate to French: {text}")
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0, google_api_key=google_api_key)
parser = StrOutputParser()

chain = prompt | llm | parser

response = chain.invoke({"text": "Good morning"})
print(response)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import CommaSeparatedListOutputParser
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env")
google_api_key = os.getenv("GOOGLE_API_KEY")

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0, google_api_key=google_api_key)

prompt = PromptTemplate.from_template("List 5 programming languages, comma-separated.")
parser = CommaSeparatedListOutputParser()
chain = prompt | llm | parser

response = chain.invoke({})
print(response)  # ['Python', 'Java', 'C++', 'JavaScript', 'Ruby']

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env")
google_api_key = os.getenv("GOOGLE_API_KEY")

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0, google_api_key=google_api_key)

class ProductInfo(BaseModel):
    name: str = Field(description="Name of the product")
    price: float = Field(description="Price in INR")

parser = PydanticOutputParser(pydantic_object=ProductInfo)

prompt = PromptTemplate(
    template="Extract product name and price from: {text}\n{format_instructions}",
    input_variables=["text"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

chain = prompt | llm | parser

response = chain.invoke({
    "text": "The Redmi Note 13 pro is available for ₹14,999."
})
print(response)  # name='Redmi Note 12' price=14999.0

In [ ]:
from langchain_classic.output_parsers.structured import ResponseSchema, StructuredOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env")
google_api_key = os.getenv("GOOGLE_API_KEY")

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0, google_api_key=google_api_key)

schemas = [
    ResponseSchema(name="company", description="Name of the company"),
    ResponseSchema(name="founder", description="Name of the founder"),
]

parser = StructuredOutputParser.from_response_schemas(schemas)

prompt = PromptTemplate(
    template="Extract company and founder from the text: {text}\n{format_instructions}",
    input_variables=["text"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

chain = prompt | llm | parser

response = chain.invoke({
    "text": "Mugil soft solutions was founded by Manikandan periyasamy in Tamil Nadu."
})
print(response)  # {'company': 'Mugil soft solutions', 'founder': 'Manikandan periyasamy'}